# Prompt Evaluation

**Live online course — instructor walkthrough notebook**

This notebook follows the course lecture notes on prompt evaluation. Each section has:
- **Lecture notes** (markdown) — what to explain on the slide/screen.
- **Demo code** — cells to run live for students to see real output.
- **🏫 During class** callouts — specific instructor actions, questions to ask, and variations to try.

By the end of the session we will have built a complete custom evaluation pipeline — dataset generation, scoring loop, and both model-based and code-based grading — in under ~150 lines of Python.

---

## Agenda

1. Why Prompt Evaluation
2. A Typical Evaluation Workflow (6 steps)
3. Generating Test Datasets
4. Running the Evaluation
5. Model Based Grading
6. Code Based Grading
7. Recap + practice exercises

## 0. Setup (do this before class starts)

1. Install dependencies:
   ```bash
   pip install anthropic python-dotenv
   ```
2. Create a file named `.env` in the same directory as this notebook containing:
   ```
   ANTHROPIC_API_KEY="sk-ant-...your-key..."
   ```
3. Add `.env` to `.gitignore` so it is never committed to version control.

> **🏫 During class:** If students attended the Intro notebook, they already have `.env` wired up. Quickly confirm `Key loaded: True` prints below — if not, pause and fix before any API cell runs, otherwise every later demo will throw.

In [ ]:
# Install packages (uncomment if not already installed)
%pip install anthropic python-dotenv

The cell below loads the API key and initializes the SDK client. It ends with three `print()` lines acting as a sanity check:

1. **SDK version** — confirms the `anthropic` package installed correctly.
2. **Model** — confirms the default workhorse (`claude-sonnet-4-6`) we'll use throughout the pipeline.
3. **Key loaded: True** — confirms `.env` was read. If this prints `False`, fix before running any API call.

In [ ]:
from dotenv import load_dotenv
import anthropic
import os

load_dotenv()  # loads ANTHROPIC_API_KEY from .env

client = anthropic.Anthropic()  # picks up ANTHROPIC_API_KEY automatically

# Default workhorse model for every demo below.
model = "claude-sonnet-4-6"

print("SDK version:", anthropic.__version__)
print("Model:", model)
print("Key loaded:", bool(os.getenv("ANTHROPIC_API_KEY")))

---
# 1. Why Prompt Evaluation

Two related but distinct practices:

| | Prompt Engineering | Prompt Evaluation |
|---|---|---|
| **What** | Writing + editing prompts so Claude understands the request | Automated testing of prompts against objective metrics |
| **Output** | A prompt | A **score** |
| **Feels like** | Copywriting | Unit testing |

### Three paths after writing a prompt

1. **Test once or twice, ship it.** 🪤 Trap — the sample size is too small to catch regressions.
2. **Hand-craft a few edge cases, tweak, ship.** 🪤 Still a trap — you're optimizing for *inputs you thought of*, not real-world distribution.
3. **Run through an evaluation pipeline with objective scoring.** ✅ Gives you an A/B comparison number, not a vibe.

### Why this matters

Engineers habitually under-test prompts. A prompt that *looks* great on two inputs fails 40% of the time on production traffic — and you won't know until users complain. Objective scoring gives you a number **before** deploy, the same way unit tests give you a number before merging.

> **🏫 During class:** Ask the room: *"Raise your hand if you've ever shipped a prompt after eyeballing 2–3 examples."* Most hands will go up. That's the problem this session solves.

### Demo: spot-the-winner between two candidate prompts

We run the *same task* through two candidate prompts (v1 vs v2). Look at the outputs and ask yourself: *which prompt is better?* If you can't answer objectively with two examples, you definitely can't answer it with 200. That's the motivation for everything that follows.

In [ ]:
task = "Write a Python function that reverses a string."

prompt_v1 = f"Please solve the following task: {task}"
prompt_v2 = (
    "You are an AWS coding assistant. Respond with ONLY raw Python code — "
    "no markdown fences, no commentary, no explanation.\n\n"
    f"Task: {task}"
)

for label, p in [("v1 (minimal)", prompt_v1), ("v2 (format-constrained)", prompt_v2)]:
    resp = client.messages.create(
        model=model,
        max_tokens=400,
        messages=[{"role": "user", "content": p}],
    )
    print(f"--- {label} ---")
    print(resp.content[0].text)
    print()

> **🏫 During class:**
> 1. Run the cell. Students will see two noticeably different outputs — v1 is usually wrapped in markdown and commentary; v2 is closer to raw code.
> 2. Ask: *"Which is better? How would you score that on a scale from 1–10?"* Let two students offer numbers. They'll disagree by 3–4 points.
> 3. Key takeaway to say out loud: *"Two humans can't agree on one example. We need **automated** grading across many examples."*

---
# 2. A Typical Evaluation Workflow

No industry-standard methodology exists. Tools range from open-source (Braintrust, promptfoo) to commercial platforms, but the core loop is always the same 6 steps:

| # | Step | What happens |
|---|---|---|
| 1 | **Write initial prompt draft** | Your baseline — the thing you'll iterate on. |
| 2 | **Create evaluation dataset** | Test inputs — 3 examples or 3,000, hand-written or LLM-generated. |
| 3 | **Generate prompt variations** | Interpolate each dataset input into the prompt template. |
| 4 | **Get LLM responses** | Feed each filled-in prompt to Claude, collect outputs. |
| 5 | **Grade responses** | A grader returns a score (e.g. 1–10). Average across the dataset. |
| 6 | **Iterate** | Modify the prompt, repeat. Compare version averages. |

### Why objective scoring beats vibes

Without a number, you cannot do A/B comparison. *"v2 feels better"* is not a commit message you can revert with confidence. *"v2 scored 8.4 vs v1 at 6.1 over 50 examples"* is.

The next four sections implement steps 2–5 one at a time. First we'll define the shared helpers everything else reuses.

The cell below defines the three shared helpers every later section reuses:

- `add_user_message` / `add_assistant_message` — thin wrappers that keep the `messages` list tidy.
- `chat(...)` — the single place we call `client.messages.create(...)`. Every demo below varies only the arguments, never the plumbing. The `thinking` parameter is disabled because later cells use assistant-message pre-fill, which is mutually exclusive with adaptive extended thinking on Sonnet 4.6 / Opus 4.7.

In [4]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})
    return messages

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})
    return messages

def chat(messages, system=None, temperature=1.0, stop_sequences=None, override_model=None):
    """Send messages to Claude and return the assistant text."""
    params = {
        "model": override_model or model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        # Sonnet 4.6 / Opus 4.7 default to adaptive extended thinking, which
        # is mutually exclusive with assistant-message pre-fill (§3, §5, §6).
        # Disable it here so every demo in the notebook behaves consistently.
        "thinking": {"type": "disabled"},
    }
    if system is not None:
        params["system"] = system
    if stop_sequences is not None:
        params["stop_sequences"] = stop_sequences
    response = client.messages.create(**params)
    return response.content[0].text

### Demo: the pipeline scaffold

Below is a map of the full pipeline — four functions we'll fill in as we go. The cell doesn't *run* a real scoring loop (the stubs raise `NotImplementedError`), but it lays out the shape. Each later section replaces exactly one stub. Keep this mental picture loaded — by §6 all four slots are filled.

In [5]:
# Pipeline scaffold — four slots, each filled in a later section.
#
# §3  generate_dataset  -> creates test cases
# §4  run_prompt        -> sends one test case through Claude
# §5  model_grade       -> scores the response (will replace the hardcoded 10)
# §6  code_grade        -> syntax validator, combined with model_grade

def generate_dataset(n):
    raise NotImplementedError("Filled in §3")

def run_prompt(prompt_template, test_case):
    raise NotImplementedError("Filled in §4")

def grade(test_case, output):
    raise NotImplementedError("Filled in §5 + §6")

def run_pipeline(prompt_template, dataset):
    results = []
    for tc in dataset:
        out = run_prompt(prompt_template, tc)
        score = grade(tc, out)
        results.append({"task": tc["task"], "output": out, "score": score})
    avg = sum(r["score"] for r in results) / len(results)
    return results, avg

print("Pipeline scaffold loaded. Stubs will be replaced in §3 -> §6.")

Pipeline scaffold loaded. Stubs will be replaced in §3 -> §6.


> **🏫 During class:**
> 1. Run the scaffold cell — show the print confirmation, no API call yet.
> 2. Walk through the four function names on screen: *"Everything we do for the rest of the hour replaces one of these four stubs."*
> 3. Ask: *"Which function do you think is the hardest to write well?"* — students usually guess `generate_dataset`; the honest answer is `grade`, and that's why §5 and §6 get dedicated treatment.

---
# 3. Generating Test Datasets

### The target prompt
We're building an **AWS code-assistance prompt** that outputs only Python, JSON config, or regex — no explanations, no markdown fences. So the dataset has to exercise all three formats.

### Two ways to get a dataset
| Approach | Cost | When to use |
|---|---|---|
| Hand-write every case | Slow, high effort | Small-scale, when you need guaranteed edge coverage |
| Ask Claude to generate them | Fast, near-free | Default. Use a **fast, cheap model** like Haiku — this is plumbing, not the system-under-test |

### Dataset shape
An array of JSON objects. Each object has at minimum a `task` property describing the user request. We'll also include a `format` field — Python, JSON, or regex — which §6's code-based grader will use to pick the right validator.

### The generation trick
Ask Claude for a JSON array → pre-fill the assistant message with the opening fence `` ```json `` → set the stop sequence to the closing fence `` ``` ``. Claude emits only the JSON content and halts. The result is directly `json.loads()`-able.

### Demo: `generate_dataset(n)`

This cell replaces the §2 stub. It asks **Haiku** (the fast, cheap model — perfect for plumbing) to produce three test cases, one for each output format. The pre-fill + stop-sequence trick cleanly extracts parseable JSON with no markdown wrapping. We save the result to `dataset.json` so §4 can load it without a re-generation round-trip.

In [ ]:
import json

def generate_dataset(n=3):
    gen_prompt = (
        f"Generate exactly {n} test cases for evaluating an AWS code-assistance prompt. "
        "Each test case should ask for one of: a short Python snippet, a JSON config block, "
        "or a regex pattern. Cover all three formats across the set.\n\n"
        "Return a JSON array of objects, each with:\n"
        '  "task"   : the user request (string)\n'
        '  "format" : one of "python", "json", "regex"\n'
    )
    messages = [
        {"role": "user", "content": gen_prompt},
        {"role": "assistant", "content": "```json"},  # opening fence pre-fill
    ]
    raw = client.messages.create(
        model="claude-haiku-4-5",  # fast + cheap for plumbing work
        max_tokens=2000,
        messages=messages,
        stop_sequences=["```"],     # closing fence stops generation
        thinking={"type": "disabled"},
    ).content[0].text
    return json.loads(raw)

dataset = generate_dataset(3)

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

print(f"Generated {len(dataset)} test cases. Saved to dataset.json\n")
for tc in dataset:
    print(f"  [{tc['format']:<6}] {tc['task']}")

> **🏫 During class:**
> 1. Run the cell. Point at the `model="claude-haiku-4-5"` line: *"Generation is plumbing, not the thing we're testing. Use the cheapest capable model."*
> 2. Open `dataset.json` in a side tab and show the file on disk. Students see the persisted artifact — this is what §4 will consume.
> 3. Variation: re-run `generate_dataset(10)` live. Watch the set expand. Ask: *"If we wanted 500 cases, what would change? (Answer: basically nothing — same loop.)"*

---
# 4. Running the Evaluation

Three tiny functions stitch the pipeline together:

| Function | Job |
|---|---|
| `run_prompt(prompt_template, test_case)` | Interpolate the test case into the prompt template, call Claude, return the text output. |
| `run_test_case(prompt_template, test_case)` | Wraps `run_prompt` and attaches a score. Returns a summary dict. |
| `run_pipeline(prompt_template, dataset)` | Loops over the dataset, calls `run_test_case` for each, returns the list. |

### Starting prompt (v1)
We begin with the most naïve template imaginable:

```
Please solve the following task: {task}
```

No format instructions. No system prompt. This is our **baseline** — the number we'll beat in later iterations.

### Known limitations of v1 (before we run it)
- Output will be **verbose** — Claude will explain every answer.
- Output will often be **wrapped in markdown fences** — bad if you want raw code.
- The grader is **hardcoded to 10** — we'll replace it in §5.

Running the pipeline surfaces these failures **as numbers**, not as hand-wavy impressions.

### Demo: run v1 end-to-end with a placeholder grader

This fills the §2 `run_prompt` stub. The grader still returns `10` for everything — that's intentional. Watch the outputs: they're verbose and markdown-wrapped, which will give us something real to grade down in §5. Typical runtime is 20–40 seconds on Sonnet for a 3-case dataset.

In [ ]:
import time

PROMPT_V1 = "Please solve the following task: {task}"

def run_prompt(prompt_template, test_case):
    filled = prompt_template.format(task=test_case["task"])
    return chat([{"role": "user", "content": filled}])

def run_test_case(prompt_template, test_case):
    output = run_prompt(prompt_template, test_case)
    score = 10  # hardcoded placeholder, replaced in §5
    return {
        "task":   test_case["task"],
        "format": test_case["format"],
        "output": output,
        "score":  score,
    }

def run_pipeline(prompt_template, dataset):
    return [run_test_case(prompt_template, tc) for tc in dataset]

start = time.time()
results_v1 = run_pipeline(PROMPT_V1, dataset)
elapsed = time.time() - start

print(f"Ran {len(results_v1)} test cases in {elapsed:.1f}s\n")
for r in results_v1:
    snippet = r["output"][:180].replace("\n", " ")
    print(f"[{r['format']:<6}] score={r['score']} — {snippet}...")

> **🏫 During class:**
> 1. Run the cell. Point out: *"Every score is 10. That's obviously wrong — the v1 prompt is terrible. But the pipeline is running end-to-end."*
> 2. Pick one result and show its full text (`print(results_v1[0]['output'])`). Most will have markdown fences and explanatory paragraphs — exactly what we said v1 wouldn't prevent.
> 3. Key takeaway: *"The loop is 10 lines. The hard part — and the next 20 minutes — is the grader."*

---
# 5. Model Based Grading

Grading is the hardest part of an evaluation pipeline. There are three broad styles:

| Grader | Signal | Flexibility | Cost | When to use |
|---|---|---|---|---|
| **Code** | Programmatic (syntax valid? length OK? keyword present?) | Low | Free | Any time the correctness criterion can be checked mechanically |
| **Model** | Another Claude call judging the output | **High** | API call per grade | Quality, tone, instruction-following — anywhere "good" is subjective |
| **Human** | A person | Highest | Slow, tedious | Final validation before prod; rarely scalable |

In practice: use **code** where you can, **model** where you can't, **human** on a spot-check sample.

### Making model grading not-useless

Model graders are inconsistent when you ask them a bare *"score 1–10"* question — they default to middling 6–7 scores on everything. Fix:

1. Ask for **strengths**, **weaknesses**, **reasoning**, and **then** the score.
2. Pin the criteria in the prompt — don't let the grader decide what "good" means.
3. Return it as JSON using the same pre-fill + stop-sequence trick from §3.

The strengths/weaknesses/reasoning scaffolding forces the grader to *justify* its score, which cuts down the tendency to pick 7 on autopilot.

### Demo: replace the hardcoded `10` with a real model grade

This cell re-grades every v1 result from §4 using a Sonnet grader. Watch what happens to the average — it'll drop sharply because v1 produces verbose, fenced output, and the grader prompt penalizes exactly that. You're watching the scoring *work*: a bad prompt gets a bad number, so a later iteration can beat it.

In [ ]:
GRADER_PROMPT = """You are judging an AWS code-assistance AI's response.

<task>
{task}
</task>

<expected_format>
{format}
</expected_format>

<response>
{output}
</response>

Score the response on two criteria:
  1. Correctness - does it actually solve the task?
  2. Directness - is it ONLY raw {format}, with NO markdown fences, NO commentary, NO explanation?

Responses that include fences or prose MUST score low on directness.

Return JSON with keys:
  "strengths" (string)
  "weaknesses" (string)
  "reasoning" (string)
  "score" (int 1-10)
"""

def model_grade(test_case, output):
    messages = [
        {"role": "user", "content": GRADER_PROMPT.format(
            task=test_case["task"], format=test_case["format"], output=output)},
        {"role": "assistant", "content": "```json"},
    ]
    raw = client.messages.create(
        model=model,
        max_tokens=600,
        messages=messages,
        stop_sequences=["```"],
        thinking={"type": "disabled"},
    ).content[0].text
    return json.loads(raw)

# Re-grade every v1 result from §4 with the real grader.
graded_v1 = []
for r in results_v1:
    verdict = model_grade({"task": r["task"], "format": r["format"]}, r["output"])
    graded_v1.append({**r, "score": verdict["score"], "reasoning": verdict["reasoning"]})

avg_v1 = sum(g["score"] for g in graded_v1) / len(graded_v1)
print(f"v1 average (model grader): {avg_v1:.2f} / 10\n")
for g in graded_v1:
    print(f"[{g['format']:<6}] score={g['score']}  — {g['reasoning'][:140]}")

> **🏫 During class:**
> 1. Run the cell. Compare the printed average to §4's implicit `10.00` — students see a concrete gap (usually around 3–6).
> 2. Pick the lowest-scoring result and read its `reasoning` field aloud: *"That's the grader explaining in English why the prompt is bad."*
> 3. Variation to try live: delete *"Responses that include fences or prose MUST score low on directness"* from `GRADER_PROMPT` and re-run. Scores jump up — showing how sensitive grading is to criteria wording.

---
# 6. Code Based Grading

Model graders have one persistent weakness: they're **judgmental, not factual**. Two Sonnet calls on the same pair of inputs can disagree by a point or two. For anything with a mechanical correctness test — *is this valid JSON? is this valid Python? does this regex compile?* — write a code grader. It's cheap, deterministic, and zero API calls.

### Three tiny validators
| Validator | Check | Score if passes / fails |
|---|---|---|
| `validate_json` | `json.loads(output)` doesn't raise | 10 / 0 |
| `validate_python` | `ast.parse(output)` doesn't raise | 10 / 0 |
| `validate_regex` | `re.compile(output)` doesn't raise | 10 / 0 |

### How to pick the right validator
This is why every dataset record has a `format` field. The validator is selected by `format`, not by inspecting the output. Guessing-from-output is brittle and defeats the point.

### Combining scores
```
final = (model_score + syntax_score) / 2
```
- `model_score` catches *quality* (did it solve the task? did it follow format instructions?).
- `syntax_score` catches *technical validity* (does the output parse?).
- Neither alone is enough.

### Prompt v2 — constrained output
We also need a **better prompt to score** so the average actually moves. v2 adds format instructions and uses pre-fill + stop-sequence to cut out markdown fences at the request level, not by post-processing.

### Demo: full pipeline — v2 prompt + model grader + code grader

This cell runs v2 through both graders and prints the combined average. Expect a clear jump above v1's number. The point isn't the specific number — it's that now we have **two numbers to compare**, which is the whole reason we built this pipeline.

In [ ]:
import ast
import re

def validate_json(s):
    try:
        json.loads(s.strip())
        return 10
    except Exception:
        return 0

def validate_python(s):
    try:
        ast.parse(s.strip())
        return 10
    except Exception:
        return 0

def validate_regex(s):
    try:
        re.compile(s.strip())
        return 10
    except Exception:
        return 0

VALIDATORS = {"python": validate_python, "json": validate_json, "regex": validate_regex}

# v2: tell the model exactly what format to produce, and use pre-fill + stop
# to strip the markdown fence at request time.
def run_prompt_v2(test_case):
    user_msg = (
        f"You are an AWS coding assistant. Respond with ONLY raw {test_case['format']} — "
        "no markdown fences, no commentary, no explanation.\n\n"
        f"Task: {test_case['task']}"
    )
    messages = [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": f"```{test_case['format']}"},
    ]
    raw = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
        stop_sequences=["```"],
        thinking={"type": "disabled"},
    ).content[0].text
    return raw

# Run v2 through both graders.
final_v2 = []
for tc in dataset:
    out = run_prompt_v2(tc)
    m_score = model_grade(tc, out)["score"]
    s_score = VALIDATORS[tc["format"]](out)
    combined = (m_score + s_score) / 2
    final_v2.append({
        "format":       tc["format"],
        "task":         tc["task"],
        "model_score":  m_score,
        "syntax_score": s_score,
        "final":        combined,
    })

avg_v2 = sum(r["final"] for r in final_v2) / len(final_v2)

print(f"v1 average (model grader only) : {avg_v1:.2f} / 10")
print(f"v2 average (model + syntax)    : {avg_v2:.2f} / 10")
print(f"                       delta   : {avg_v2 - avg_v1:+.2f}\n")

for r in final_v2:
    print(f"[{r['format']:<6}] model={r['model_score']}  syntax={r['syntax_score']}  final={r['final']:.1f}")

> **🏫 During class:**
> 1. Run the cell. Point to the v1-vs-v2 delta: *"That number — not the vibes — is why we'd ship v2."*
> 2. If a row shows `syntax=0`, open that row's `task` and look at the output together. Almost always: Claude still sneaked in a commented preamble that broke the parser. This is the grader earning its keep.
> 3. Variation to try live: add a **v3** that uses a system prompt (`"Output must parse. No exceptions."`) and re-run the comparison. Does it beat v2? Sometimes yes, sometimes no — which is exactly the surprise factor that makes this work worth doing.
> 4. Key takeaway: *"Code graders are free and deterministic. Use them wherever you can. Reserve model graders for the subjective layer on top."*

---
# 7. Recap + practice exercises

### Recap (run through these out loud)
- **Engineering vs Evaluation:** one writes prompts; the other *tests* them with numbers.
- **The 6-step loop:** draft → dataset → variations → responses → grade → iterate.
- **Datasets:** generate them with the cheapest capable model (Haiku). Start with 3 cases, grow to thousands.
- **Three grader types:** code (cheap, deterministic), model (flexible, subjective), human (expensive, ground truth).
- **Model grader trick:** ask for strengths + weaknesses + reasoning **before** the score. Raw *"give me 1–10"* produces middle scores on everything.
- **Code grader trick:** put the expected `format` in every dataset record so the validator is chosen by data, not guessed from output.
- **Combining:** `(model_score + syntax_score) / 2` captures *quality* and *technical validity* together.
- **The point of it all:** A/B comparison of prompt versions as **numbers**, not vibes.

### Exercises (do the first in class, assign the rest)
1. **Bigger dataset:** Re-run §3 with `n=20`, then re-run the v1 vs v2 comparison. Does the average stabilize? Do the v1/v2 deltas shift?
2. **Stricter code grader:** Add a `length_score` that gives 10 when the output is under 300 characters, 0 otherwise. Combine all three graders with weights of your choosing.
3. **v3 prompt:** Write a v3 that adds a system prompt (e.g. *"Output must be machine-parseable. Omit all commentary."*) and beat v2's score.
4. **Grader disagreement audit:** Run the model grader twice on the same output. Calculate the variance across the dataset. At what point is model grading unreliable enough to require a human spot-check?
5. **Cost-aware scoring:** Extend `run_test_case` to record `usage.input_tokens + usage.output_tokens`. Add average cost to the recap line alongside average score — now you can pick the winning prompt on quality-per-dollar, not just quality.

In [ ]:
# Exercise 1 scaffold — uncomment and finish live in class.
#
# big_dataset = generate_dataset(20)
# with open("dataset.json", "w") as f:
#     json.dump(big_dataset, f, indent=2)
#
# def score_prompt_v1(ds):
#     results = [run_test_case(PROMPT_V1, tc) for tc in ds]
#     graded  = [{**r, "score": model_grade(r, r["output"])["score"]} for r in results]
#     return sum(g["score"] for g in graded) / len(graded)
#
# def score_prompt_v2(ds):
#     finals = []
#     for tc in ds:
#         out = run_prompt_v2(tc)
#         m = model_grade(tc, out)["score"]
#         s = VALIDATORS[tc["format"]](out)
#         finals.append((m + s) / 2)
#     return sum(finals) / len(finals)
#
# print("v1:", score_prompt_v1(big_dataset))
# print("v2:", score_prompt_v2(big_dataset))